# **Optimización TP1**: Perfil de Desempeño

Diseñado por Dolan y Moré en 2001, es una herramienta que se emplea para comparar la aptitud de distintos algoritmos para resolver un conjunto de problemas.

Dados un conjunto de problemas $P$ y un conjunto de algoritmos $S$, definimos como $c_{s,p}$ el costo de resolver el problema $p \in P$ con el algoritmo $s \in S$. Si el algoritmo $s$ no puede resolver el problema $p$, se define $c_{s,p} = c_{max}$ con $c_{max} \in \mathbb{R}$ un valor adecuado de manera tal que:

$$c_{s,p} = c_{max} \Leftrightarrow \textbf{el algoritmo } s \textbf{ no resuelve el problema } p$$

Se asume que al menos un algoritmo resuelve el problema $p$. Se define la tasa de desempeño relativa como:

$$r_{s,p} = \frac{c_{s,p}}{\min\{c_{j,p}: j \in S\}}$$

Observar que $r_{s,p} \geq 1$ y, si $r_{s,p} = 1$, entonces el algoritmo $s$ es uno de los mejores (o el mejor) para resolver el problema $p$. Mientras mayor sea $r_{s,p}$, peor es el desempeño de $s$ al resolver $p$. Finalmente, se define la función de desempeño $\rho_s : [1, \infty) \to [0, 1]$ para el algoritmo $s$ como:

$$\rho_s(\tau) = \frac{\#\{p \in P : r_{s,p} \leq \tau\}}{\#P}$$

Así, $\rho_s(\tau)$ es el porcentaje de problemas que el algoritmo $s$ resuelve con hasta $\tau$ veces el costo del algoritmo más eficiente. Por ejemplo, $\rho_s(1)$ es la proporción de problemas que el algoritmo $s$ resuelve con el menor costo.

Si se define:

$$r_{max} = \max_{s \in S,\, p \in P\,:\, c_{s,p} < M} r_{s,p}$$

se tiene que $\rho_s(r_{max})$ es el número de problemas resueltos por el algoritmo $s$. El valor $\rho_s(1)$ se denomina la eficiencia de $s$ y $\rho_s(r_{max})$ es su robustez.

La definición del costo está relacionada a la medida de desempeño que se quiera estudiar. Se puede considerar como **costo** al **tiempo que necesita el algoritmo para resolver el problema, a la cantidad de iteraciones, a la cantidad de veces que es evaluada la función objetivo**, etc.

Para analizar el perfil de desempeño se suelen graficar en conjunto las funciones $\rho_s$ para cada $s \in S$.

## **Ejercicios**: Análisis de desempeño

In [45]:
using Pkg
Pkg.add(["DataFrames", "LinearAlgebra", "Plots"])

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`


In [46]:
include("benchmarks.jl")
using LinearAlgebra, Plots, DataFrames

### **Ejercicio 5**: Métodos de descenso

In [47]:
# Método de descenso por gradiente con paso constante

function gradiente_constante(x₀, f, ∇f, η=0.1, max_iter=1000)
    # Condiciones iniciales
    xₖ = x₀
    iter = 0
    status = "MAX_ITER"

    # Iteraciones
    while iter < max_iter
        gₖ = ∇f(xₖ)
        
        # Criterio de parada
        if norm(gₖ) < 1e-6
            status = "CONVERGIO"
            break
            
        end
        # Actualización
        xₖ = xₖ - η .* gₖ
        iter += 1
        
    end
    return status, iter, norm(∇f(xₖ)), xₖ
    
end

gradiente_constante (generic function with 3 methods)

In [54]:
# Método de descenso por gradiente con búsqueda lineal (Método de Armijo)

function gradiente_armijo(x₀, f, ∇f, max_iter=1000)
    # Condiciones iniciales
    xₖ = x₀
    αₖ = 1/2
    iter = 0
    status = "MAX_ITER"

    # Iteraciones
    while iter < max_iter
        # Criterio de parada
        gₖ = ∇f(xₖ)
        if norm(gₖ) < 1e-6
            status = "CONVERGIO"
            break
        end

        # Armijo
        h(x)  = f(xₖ - x .* gₖ)  
        dh₀   = -norm(gₖ)^2
        
        while true
            println("Armijo: αₖ = $αₖ")
            # Decrecimiento suficiente
            if (h(αₖ) > h(0) + dh₀/2)
                αₖ = αₖ/2
                
            # Pasos no muy cortos
            elseif (h(2αₖ) < h(0) + αₖ*dh₀)
                αₖ = αₖ*10
                
            # Cumple Armijo
            else
                break
            end
        end    
        
        # Actualización
        xₖ = xₖ - αₖ .* gₖ
        iter += 1
        
    end
    return status, iter, norm(∇f(xₖ)), xₖ
    
end

gradiente_armijo (generic function with 2 methods)

In [55]:
# Método de Newton

function newton(x₀, f, ∇f, Hf, max_iter=1000)
    # Condiciones iniciales
    xₖ = x₀
    iter = 0
    status = "MAX_ITER"

    # Iteraciones
    while iter < max_iter
        # Criterio de parada
        gₖ = ∇f(xₖ)
        if norm(gₖ) < 1e-6
            status = "CONVERGIO"
            break
        end
        
        # Intento descomposición de Cholesky
        C = cholesky(Hf(xₖ), check=false)

        # Si lo logro resuelvo el problema lineal
        if issuccess(C)         
            dₖ = C \ (-gₖ)   
            xₖ = xₖ + dₖ
            iter += 1 
            
        # Si no, no converge  
            status = "NO_CONVERGIO"
            break
            
        end
    end
    return status, iter, norm(∇f(xₖ)), xₖ
    
end

newton (generic function with 2 methods)

In [56]:
# Método de gradientes conjugados no-lineal (Método Fletcher–Reeves)

function gradientes_conjugados(x₀, f, ∇f, max_iter=1000)
    # Condiciones iniciales
    xₖ = x₀
    gₖ = ∇f(x₀)
    αₖ = 1
    dₖ = -gₖ
    n      = length(x₀)
    iter   = 0
    status = "MAX_ITER"

    # Iteraciones
    while iter < max_iter
        # Criterio de parada
        gₖ = ∇f(xₖ)
        if norm(gₖ) < 1e-6
            status = "CONVERGIO"
            break
        end

        # Goldstain
        h(x)  = f(xₖ + x .* dₖ)  
        dh₀   = ∇f(xₖ)' * dₖ
        
        while true
            # Decrecimiento suficiente
            if (h(αₖ) > h(0) + dh₀/2)
                αₖ = αₖ/2
                
            # Pasos no muy cortos
            elseif (h(αₖ) < h(0) + αₖ*dh₀/2)
                αₖ = αₖ*10
                
            # Cumple Goldstein
            else
                break
            end
        end

        # Actualización
        xₖ = xₖ + αₖ .* dₖ
        βₖ = (norm(∇f(xₖ))/norm(gₖ))^2
        gₖ = ∇f(xₖ)
        dₖ = (iter%n == n-1) ? -gₖ : -gₖ + βₖ .* dₖ
        iter += 1
         
    end
    return status, iter, norm(∇f(xₖ)), xₖ
    
end

gradientes_conjugados (generic function with 2 methods)

### **Ejercicio 6**: Benchmarks

In [57]:
# Puntos iniciales
x0_2d = [1.0, 1.0]
x0_rb = [-1.2, 1.0]
x0_wd = [-3.0, -1.0, -3.0, -1.0]
x0_fr = [0.5, -2.0]

# Tabla de funciones
benchmarks = [
    ("Sphere",      sphere,      sphere_g,      sphere_h,      x0_2d),
    ("Rosenbrock",  rosenbrock,  rosenbrock_g,  rosenbrock_h,  x0_rb),
    ("Rastrigin",   rastrigin,   rastrigin_g,   rastrigin_h,   x0_2d),
    ("Wood",        wood,        wood_g,        wood_h,        x0_wd),
    ("Froth",       froth,       froth_g,       froth_h,       x0_fr),
]

# Tabla de métodos
metodos = [
    ("Grad. Constante",  (f, ∇f, Hf, x0) -> gradiente_constante(x0, f, ∇f)),
    ("Grad. Armijo",     (f, ∇f, Hf, x0) -> gradiente_armijo(x0, f, ∇f)),
    ("Newton",           (f, ∇f, Hf, x0) -> newton(x0, f, ∇f, Hf)),
    ("Grad. Conjugados", (f, ∇f, Hf, x0) -> gradientes_conjugados(x0, f, ∇f)),
]

# Tabla de resultados
resultados = DataFrame(
    Funcion  = String[],
    Metodo   = String[],
    Status   = String[],
    Iters    = Int[],
    NormGrad = Float64[],
    xFinal   = Vector{Float64}[]
)

Row,Funcion,Metodo,Status,Iters,NormGrad,xFinal
,String,String,String,Int64,Float64,Array…


In [58]:
# Evaluaciones
for (nombre, f, ∇f, Hf, x0) in benchmarks
    for (nombre_met, metodo) in metodos
        print("Ejecutando $nombre_met en $nombre... ")
        s, i, g, xk = metodo(f, ∇f, Hf, x0)
        push!(resultados, (nombre, nombre_met, s, i, g, xk))
        println("$s en $i iteraciones")
    end
end

Ejecutando Grad. Constante en Sphere... CONVERGIO en 67 iteraciones
Ejecutando Grad. Armijo en Sphere... Armijo: αₖ = 0.5
Armijo: αₖ = 0.25
Armijo: αₖ = 0.125
Armijo: αₖ = 0.0625
Armijo: αₖ = 0.03125
Armijo: αₖ = 0.015625
Armijo: αₖ = 0.0078125
Armijo: αₖ = 0.00390625
Armijo: αₖ = 0.001953125
Armijo: αₖ = 0.0009765625
Armijo: αₖ = 0.00048828125
Armijo: αₖ = 0.000244140625
Armijo: αₖ = 0.0001220703125
Armijo: αₖ = 6.103515625e-5
Armijo: αₖ = 3.0517578125e-5
Armijo: αₖ = 1.52587890625e-5
Armijo: αₖ = 7.62939453125e-6
Armijo: αₖ = 3.814697265625e-6
Armijo: αₖ = 1.9073486328125e-6
Armijo: αₖ = 9.5367431640625e-7
Armijo: αₖ = 4.76837158203125e-7
Armijo: αₖ = 2.384185791015625e-7
Armijo: αₖ = 1.1920928955078125e-7
Armijo: αₖ = 5.960464477539063e-8
Armijo: αₖ = 2.9802322387695312e-8
Armijo: αₖ = 1.4901161193847656e-8
Armijo: αₖ = 7.450580596923828e-9
Armijo: αₖ = 3.725290298461914e-9
Armijo: αₖ = 1.862645149230957e-9
Armijo: αₖ = 9.313225746154785e-10
Armijo: αₖ = 4.656612873077393e-10
Armijo

Excessive output truncated after 524293 bytes.

Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0.0
Armijo: αₖ = 0

LoadError: InterruptException: